# Data Integrity Compiler, Milestone 1 validation

Run this notebook on a Kaggle CPU runtime. The workload is DuckDB and does not need a GPU. Add `data-integrity-compiler-source.zip` as a notebook input, enable Internet, and run all cells. This notebook downloads official TLC files, checks their SHA-256 values, runs clean controls and the M1 fault experiment, independently reconciles the January Parquet outputs, and creates a downloadable results archive.

The report distinguishes pipeline execution from integrity outcome. A blocked fault run is expected.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, urllib.request, zipfile
from decimal import Decimal

inputs = Path("/kaggle/input")
archives = list(inputs.rglob("data-integrity-compiler-source.zip"))
if not archives:
    raise FileNotFoundError("Add data-integrity-compiler-source.zip as a Kaggle notebook input dataset")
ROOT = Path("/kaggle/working/data-integrity-compiler")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
with zipfile.ZipFile(archives[0]) as z: z.extractall(ROOT)
print("Project extracted to", ROOT)
print("Python", sys.version)

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-lock.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)], check=True)
print("Pinned project dependencies installed")

In [ ]:
raw = ROOT / "data" / "raw"
raw.mkdir(parents=True, exist_ok=True)
expected = {
    "yellow_tripdata_2024-01.parquet": "c4d59da7bbc8abaeeeb1727947ee93d9891a71acb42854bd80db1571b2030510",
    "yellow_tripdata_2024-02.parquet": "c76c43c18c6c6664080dd920baab4928988d5786a6b65980792ca7cd796f9f20",
    "yellow_tripdata_2025-01.parquet": "9af277e4c0d3f9deb30644da822981e1e7df6af58313170fd3aa8a474485488a",
    "taxi_zone_lookup.csv": "1a99e105092230f8620f301edcca7f80d3080642ff404d28ed957d3fa222c8ed",
}
urls = {name: ("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv" if name == "taxi_zone_lookup.csv" else "https://d37ci6vzurychx.cloudfront.net/trip-data/" + name) for name in expected}
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()
for name,url in urls.items():
    target=raw/name
    if not target.exists(): urllib.request.urlretrieve(url,target)
    actual=sha256(target)
    assert actual == expected[name], f"Checksum mismatch for {name}: {actual}"
print("All official source checksums verified")

In [ ]:
def dic(*args):
    result=subprocess.run([sys.executable,"-m","dic.cli",*map(str,args)],cwd=ROOT,text=True,capture_output=True)
    if result.returncode: raise RuntimeError(result.stderr[-4000:] or result.stdout[-4000:])
    return json.loads(result.stdout)
for month in ("2024-01","2024-02","2025-01"):
    meta=dic("acquire","--month",month)
    print(month, meta["schema_classification"], meta["added_columns"])
    assert meta["schema_classification"] == "compatible"
manifest=json.loads((ROOT/"data/source-manifest.json").read_text())
assert manifest["months"]["2025-01"]["added_columns"] == ["cbd_congestion_fee"]
assert manifest["months"]["2024-02"]["added_columns"] == []

In [ ]:
controls=ROOT/"data/clean-controls"
controls.mkdir(parents=True,exist_ok=True)
reports={}
for month in ("2024-02","2025-01","2024-01","2024-01"):
    report=dic("run","--month",month)
    assert report["outcome"] == "passed", month
    reports.setdefault(month,[]).append(report)
    suffix="-repeat" if len(reports[month]) == 2 else ""
    (controls/f"{month}{suffix}.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    print(month, report["run_id"], report["runtimes_seconds"])
jan_clean=reports["2024-01"][0]
jan_repeat=reports["2024-01"][1]
assert jan_clean["logical_output_hashes"] == jan_repeat["logical_output_hashes"]
assert all(all(c["passed"] for c in r["checks"]) for rs in reports.values() for r in rs)
print("Clean controls passed; January logical fingerprints repeat")

In [ ]:
injected=dic("inject-m1","--month","2024-01","--key","1")
fault=dic("run","--month","2024-01","--lookup",injected["lookup"],"--reference-lookup",raw/"taxi_zone_lookup.csv","--fault-id",injected["fault_id"])
assert fault["outcome"] == "blocked"
(ROOT/"data/fault_report.json").write_text(json.dumps(fault,indent=2),encoding="utf-8")
(ROOT/"data/clean_report.json").write_text(json.dumps(jan_clean,indent=2),encoding="utf-8")
evaluated=dic("evaluate","--report",ROOT/"data/runs"/fault["run_id"]/"report.json","--ground-truth",injected["ground_truth"])
print("Fault evaluation:",evaluated)
assert evaluated["fault_id_matches"] and evaluated["key_matches"] and evaluated["invariant_detected"]
assert fault["publication"]["pointer_before"] == fault["publication"]["pointer_after"] == jan_repeat["run_id"]
print("Fault blocked and prior clean publication pointer was preserved")

In [ ]:
# Independent comparison reads the saved Parquet outputs, not the detector ground truth.
subprocess.run([sys.executable,str(ROOT/"experiments/reconcile_reports.py"),"--root",str(ROOT)],cwd=ROOT,check=True)
reconciliation=json.loads((ROOT/"data/jan-2024-reconciliation.json").read_text())
assert reconciliation["expected_row_inflation"] == reconciliation["observed_row_inflation"]
assert reconciliation["report_aggregate_deltas_match_parquet"]
assert reconciliation["average_fares_unchanged_in_all_affected_groups"]
assert fault["expected_observed"]["row_inflation"] == reconciliation["observed_row_inflation"]
assert reconciliation["clean_run_id"] == jan_clean["run_id"]
assert reconciliation["contract_sha256"] == jan_clean["contract_hash"]
assert all(sha256(raw/name)==expected[name] for name in expected)
print(json.dumps(reconciliation,indent=2))

In [ ]:
# Export compact reports plus the January clean and blocked-fault Parquet evidence.
export=Path("/kaggle/working/dic-m1-validation-results.zip")
keep=[ROOT/"data/source-manifest.json",ROOT/"data/jan-2024-reconciliation.json",ROOT/"data/clean_report.json",ROOT/"data/fault_report.json",*controls.glob("*.json")]
keep.append(Path(injected["ground_truth"]))
# Include the exact clean run referenced by reconciliation, its manifest, the repeat run, and the fault run.
for run_id in (jan_clean["run_id"],jan_repeat["run_id"],fault["run_id"]):
    d=ROOT/"data/runs"/run_id
    keep.extend(p for p in d.iterdir() if p.is_file() and p.suffix in (".parquet",".json",".csv"))
clean_manifest=json.loads((ROOT/"data/runs"/jan_clean["run_id"]/"manifest.json").read_text())
assert clean_manifest["contract_sha256"] == reconciliation["contract_sha256"]
published=ROOT/"data/published"
if (published/"CURRENT").exists(): keep.append(published/"CURRENT")
with zipfile.ZipFile(export,"w",zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for p in keep:
        if p.exists(): z.write(p,p.relative_to(ROOT).as_posix())
print("Download this output:",export)
print("Validation complete")